# Modèle Word2Vec et similarité cosinus

### Lecture du dataset nettoyé

In [2]:
import pandas as pd

df = pd.read_csv("../../datasets/clean/perfumes_clean.csv")

In [3]:
df.head()


,id,name,house,url,family,secondary_family,character,top_notes,heart_notes,base_notes,...,times_of_day,season_spring,season_summer,season_autumn,season_winter,time_day,time_evening,time_night,description,text_full
0,ysl-libre,Libre,Yves Saint Laurent,https://www.scento.com/perfumes/libre-yves-sai...,White Bloom,NaN,floral; creamy; fresh; citrusy,mandarin; neroli; lavender,orange blossom; jasmine sambac,musk; vanilla,...,day; evening,1,1,1,0,1,1,0,Mandarin and neroli brighten a creamy bloom of...,Mandarin and neroli brighten a creamy bloom of...
1,guerlain-shalimar,Shalimar,Guerlain,https://www.scento.com/perfumes/shalimar-eau-d...,Petal Garden,Velvet Sweet,floral; sweet; warm; powdery,floral notes; bergamot,iris; jasmine; rose,vanilla; opoponax; tonka bean,...,day; evening,1,1,1,0,1,1,0,Floral notes and a flash of bergamot sit above...,Floral notes and a flash of bergamot sit above...
2,tom-ford-oud-wood,Oud Wood,Tom Ford,https://www.scento.com/perfumes/oud-wood-tom-ford,Spice Route,NaN,warm; spicy; sweet; earthy,NaN,NaN,NaN,...,evening; night,0,0,1,1,0,1,1,Smooth woods and spice: cardamom and pink pepp...,Smooth woods and spice: cardamom and pink pepp...
3,creed-aventus,Aventus,Creed,https://www.scento.com/perfumes/aventus-creed,Orchard Fruit,Wood & Root,fruity; earthy; citrusy; floral,lemon; pink pepper; apple; bergamot; blackcurrant,pineapple; jasmine; patchouli,birch; ambroxan; cedarwood; oakmoss; musk,...,day; evening,1,1,1,0,1,1,0,Sparkling lemon and blackcurrant open onto pin...,Sparkling lemon and blackcurrant open onto pin...
4,mfk-baccarat-rouge-540,Baccarat Rouge 540,Maison Francis Kurkdjian,https://www.scento.com/perfumes/baccarat-rouge...,Wood & Root,White Bloom,woody; warm; floral; spicy,jasmine; saffron,cedarwood,ambergris,...,day; evening,1,0,1,1,1,1,0,"Jasmine and saffron open airy and golden, held...","Jasmine and saffron open airy and golden, held..."


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1479 entries, 0 to 1478
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                1479 non-null   object 
 1   name              1479 non-null   object 
 2   house             1479 non-null   object 
 3   url               1479 non-null   object 
 4   family            1479 non-null   object 
 5   secondary_family  1075 non-null   object 
 6   character         1479 non-null   object 
 7   top_notes         1388 non-null   object 
 8   heart_notes       1388 non-null   object 
 9   base_notes        1388 non-null   object 
 10  notes             1479 non-null   object 
 11  concentration     1479 non-null   object 
 12  intensity         1228 non-null   float64
 13  seasons           1479 non-null   object 
 14  times_of_day      1479 non-null   object 
 15  season_spring     1479 non-null   int64  
 16  season_summer     1479 non-null   int64  


Le dataset propre contient désormais 24 colonnes. Les colonnes de saisons et de moments de la journée sont en binaires, donc seront exploitables dans le questionnaire lors de la création de l'interface. La colonne text_full contient toutes les données de la ligne dans une colonne.

### Tokénisation

On commence par l'étape de tokénisation. Word2Vec nécessite des données sous forme de listes de mots et non de phrase complète. Il s'agit donc de découper chaque description de la colonne texte_full en liste de mots. Pour plus d'uniformité, nous allons aussi en profité pour mettre tous les caractères en minuscule. 

Après une première vérification des résultats, certains mots étaient tokénisés avec des virgules ou des points. On retire donc également la ponctuation.

In [5]:
# découpage du texte en mot
corpus = df['text_full'].apply(lambda texte: texte.lower().split())

# retirage de la ponctuation
corpus = df["text_full"].apply(lambda texte: texte.lower().replace(",", "").replace(".", "").replace(":", "").split())

print('Texte original : ', df['text_full'].iloc[0])
print('Texte tokénisé : ', corpus.iloc[0])

Texte original :  Mandarin and neroli brighten a creamy bloom of orange blossom and jasmine sambac, settling into a quiet base of musk and vanilla. Olfactory family: White Bloom. Character: floral, creamy, fresh, citrusy. Top notes: mandarin, neroli, lavender. Heart notes: orange blossom, jasmine sambac. Base notes: musk, vanilla. Seasons: spring, summer, autumn. Time of day: day, evening.
Texte tokénisé :  ['mandarin', 'and', 'neroli', 'brighten', 'a', 'creamy', 'bloom', 'of', 'orange', 'blossom', 'and', 'jasmine', 'sambac', 'settling', 'into', 'a', 'quiet', 'base', 'of', 'musk', 'and', 'vanilla', 'olfactory', 'family', 'white', 'bloom', 'character', 'floral', 'creamy', 'fresh', 'citrusy', 'top', 'notes', 'mandarin', 'neroli', 'lavender', 'heart', 'notes', 'orange', 'blossom', 'jasmine', 'sambac', 'base', 'notes', 'musk', 'vanilla', 'seasons', 'spring', 'summer', 'autumn', 'time', 'of', 'day', 'day', 'evening']


In [6]:
print("Nombre de documents :", len(corpus))
print("Nombre de mots du premier parfum :", len(corpus.iloc[0]))

longueurs = corpus.apply(len)
print("Nombre minimum de mots :", longueurs.min())
print("Nombre maximum de mots :", longueurs.max())
print("Nombre moyen de mots :", round(longueurs.mean(), 2))

Nombre de documents : 1479
Nombre de mots du premier parfum : 55
Nombre minimum de mots : 29
Nombre maximum de mots : 86
Nombre moyen de mots : 58.48


Après tokénisation, on obtient 1479 corpus, soit un par parfum. Chaque corpus contient en moyenne 58,48 mots, avec 29 en minimum et 86 en maximum

### Suppression des stop words

Comme vu en cours, l'étape suivante est de retirer les stop words pour ne pas garder des mots inutiles qui ne joue pas sur le sens de la phrase.

In [7]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

corpus = corpus.apply(lambda mots: [mot for mot in mots if mot not in ENGLISH_STOP_WORDS])

In [8]:
print('Texte après suppression des stop words : ', corpus.iloc[0])

print('Moyenne de mots avant : ', round(longueurs.mean(), 2))
print('Moyenne de mots après : ', round(corpus.apply(len).mean(), 2))

Texte après suppression des stop words :  ['mandarin', 'neroli', 'brighten', 'creamy', 'bloom', 'orange', 'blossom', 'jasmine', 'sambac', 'settling', 'quiet', 'base', 'musk', 'vanilla', 'olfactory', 'family', 'white', 'bloom', 'character', 'floral', 'creamy', 'fresh', 'citrusy', 'notes', 'mandarin', 'neroli', 'lavender', 'heart', 'notes', 'orange', 'blossom', 'jasmine', 'sambac', 'base', 'notes', 'musk', 'vanilla', 'seasons', 'spring', 'summer', 'autumn', 'time', 'day', 'day', 'evening']
Moyenne de mots avant :  58.48
Moyenne de mots après :  48.3


Avec la suppression des stop words, on finit avec 48,3 mots en moyenne au liey de 58,48 avant.

### Suppression des mots de structure

On retire également les mots de structure pour ne pas fausser l'apprentissage des descriptions.

In [18]:
mots_inutiles = ['olfactory', 'family', 'character', 'notes', 'seasons', 'time', 'day', 'heart', 'base']

corpus = corpus.apply(lambda mots: [mot for mot in mots if mot not in mots_inutiles])

In [19]:
print(corpus.iloc[0])
print('Moyenne de mots après nettoyage : ', round(corpus.apply(len).mean(), 2))

['mandarin', 'neroli', 'brighten', 'creamy', 'bloom', 'orange', 'blossom', 'jasmine', 'sambac', 'settling', 'quiet', 'musk', 'vanilla', 'white', 'bloom', 'floral', 'creamy', 'fresh', 'citrusy', 'mandarin', 'neroli', 'lavender', 'orange', 'blossom', 'jasmine', 'sambac', 'musk', 'vanilla', 'spring', 'summer', 'autumn', 'evening']
Moyenne de mots après nettoyage :  35.83


On passe désormais à 35,83 mots en moyenne.

### Vérification du vocabulaire

In [20]:
from collections import Counter

# on compte les occurrences de chaque mot
frequences = Counter(mot for parfum in corpus for mot in parfum)

print('Nombre de mots différents : ', len(frequences))
print('\nLes 20 mots les plus fréquents : ')
print(frequences.most_common(20))

Nombre de mots différents :  1239

Les 20 mots les plus fréquents : 
[('evening', 1445), ('autumn', 1442), ('spring', 1415), ('warm', 1192), ('floral', 1176), ('musk', 1171), ('vanilla', 1059), ('sweet', 997), ('bergamot', 992), ('amber', 949), ('summer', 880), ('patchouli', 867), ('jasmine', 866), ('rose', 832), ('sandalwood', 748), ('wood', 701), ('orange', 672), ('fresh', 672), ('cedarwood', 622), ('&', 617)]


In [21]:
mots_uniques = sum(1 for frequence in frequences.values() if frequence == 1)

print('Mots apparaissant une seule fois : ', mots_uniques)
print("Pourcentage de mot apparaissant qu'une seule fois : ", round(mots_uniques / len(frequences) * 100, 2), "%")

Mots apparaissant une seule fois :  407
Pourcentage de mot apparaissant qu'une seule fois :  32.85 %


Certains mots sont très fréquents, mais 32,8% n'apparaissent qu'une seule fois. Il peut donc être intéressant de quand même garder les mots rares pour la vectorisation.

### Entraînement de Word2Vec

In [22]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=1, # on garde tous les mots
    workers=4,
    sg=1,
    epochs=100
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [23]:
print('Nombre de mots dans le modèle : ', len(model.wv))
print('Dimension des vecteurs : ', model.wv.vector_size)

Nombre de mots dans le modèle :  1239
Dimension des vecteurs :  100


On retrouve 1239 mots et des vecteurs de 100 dimensions.

In [24]:
model.wv.most_similar('jasmine', topn=10)

[('patchouli', 0.5874959230422974),
 ('rose', 0.5647716522216797),
 ('vanilla', 0.5346561074256897),
 ('musk', 0.5253255367279053),
 ('float', 0.5245589017868042),
 ('jammy', 0.5176842212677002),
 ('surrounded', 0.5162547826766968),
 ('women', 0.5148064494132996),
 ('appear', 0.510486364364624),
 ('bergamot', 0.5072787404060364)]

In [16]:
model.wv.most_similar('musk', topn=10)

[('sandalwood', 0.5944100022315979),
 ('vanilla', 0.5881373286247253),
 ('amber', 0.5748721957206726),
 ('transparent', 0.5286200046539307),
 ('base', 0.5237839221954346),
 ('jasmine', 0.49780288338661194),
 ('kind', 0.4954249858856201),
 ('vetiver', 0.49366167187690735),
 ('nectarine', 0.4932209849357605),
 ('wood', 0.4897867441177368)]

In [17]:
model.wv.most_similar('vanilla', topn=10)

[('musk', 0.5881373286247253),
 ('amber', 0.5880957841873169),
 ('sandalwood', 0.5835381150245667),
 ('bean', 0.5292457938194275),
 ('sweet', 0.5238098502159119),
 ('amber-toned', 0.5216721296310425),
 ('brûlée', 0.5173020958900452),
 ('patchouli', 0.5154687762260437),
 ('tonka', 0.5049101710319519),
 ('cotta', 0.4985639154911041)]

Les scores de similarité varient entre 0,5 et 0,6 environ. Par exemple, pour jasmine, les senteurs proches sont patchouli, jasmin et orange. Pour vanilla, le musk et l'ambre sont proches. La vanille est d'ailleurs retrouvé aussi dans le top 10 du musk, ce qui est cohérent.

### Création des vecteurs de parfums

In [27]:
import numpy as np

def moyenne_parfum(liste):
    vecteurs = [model.wv[mot] for mot in liste]
    return np.mean(vecteurs, axis=0)

On commence par faire une moyenne des vecteurs de chaque mots d'une ligne.

In [28]:
vecteur_test = moyenne_parfum(corpus.iloc[0])

print(vecteur_test)
print('Dimension du vecteur : ', vecteur_test.shape)

[-0.04196388  0.26649898 -0.07771814 -0.02442721 -0.00100078 -0.15248564
  0.28516376  0.20082515 -0.24811435 -0.14332756 -0.25821435 -0.11608399
  0.24561203 -0.07547672  0.1306925  -0.17876479  0.09716827 -0.21812533
  0.13371544 -0.25054896  0.25500494 -0.08408249  0.16627021  0.04312704
 -0.20156492 -0.10913577 -0.00133924 -0.0526946  -0.1494355  -0.00059707
  0.0229512  -0.09941798  0.06099741 -0.09574887  0.03448534  0.11983442
 -0.06248099 -0.11320784  0.02372379 -0.1754818   0.03168611 -0.4796534
  0.12642768  0.11995169  0.20539333 -0.3729165  -0.15949503 -0.3770469
  0.02985936  0.05843216  0.22764266 -0.08005974  0.09872916 -0.05532388
  0.01252372  0.11069976  0.15433045 -0.0226667  -0.3616202  -0.00535636
  0.04659067 -0.2832085   0.18375868  0.03980742 -0.08747157  0.07006331
 -0.15793636  0.300398   -0.22652484  0.17064771 -0.09938464 -0.05393962
  0.2905206   0.08646518  0.10549425  0.05528796  0.05566911  0.0681703
 -0.08299711  0.19521022 -0.11284912  0.00171122 -0.16

La fonction semble fonctionner : on obtient bien un vecteur de 100 nombres

In [29]:
# On applique cela à tout les parfums

vecteurs_parfums = np.array([
    moyenne_parfum(mots) for mots in corpus
])

print("Dimensions de la matrice :", vecteurs_parfums.shape)

Dimensions de la matrice : (1479, 100)


Après application sur tout les parfums, on fini bien avec 1479 vecteurs de dimension 100, soit un vecteur par lignes du dataset.

### Similarité cosinus entre parfums

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# comparaison de chaque parfum avec les autres
similarites = cosine_similarity(vecteurs_parfums)

/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


### Première recommendation sur base d'un parfum

In [31]:
def recommander_parfums(indice, n=5):
    scores = similarites[indice]

    indices = scores.argsort()[::-1][1:n+1]

    resultats = df.iloc[indices][["name", "house"]].copy()
    resultats["similarite"] = scores[indices]

    return resultats

# test
print("Parfum choisi :", df.iloc[0]["name"])
recommander_parfums(0)

Parfum choisi : Libre


,name,house,similarite
752,Connoisseur Woman,Armaf,0.970985
337,Mon Guerlain,Guerlain,0.963808
1445,Valentino Voce Viva Intensa,Valentino,0.960482
1226,La vie est belle L'Éclat,Lancôme,0.951628
799,Mirage Woman,Armaf,0.947083


In [33]:
print(df.iloc[0])

id                                                          ysl-libre
name                                                            Libre
house                                              Yves Saint Laurent
url                 https://www.scento.com/perfumes/libre-yves-sai...
family                                                    White Bloom
secondary_family                                                  NaN
character                              floral; creamy; fresh; citrusy
top_notes                                  mandarin; neroli; lavender
heart_notes                            orange blossom; jasmine sambac
base_notes                                              musk; vanilla
notes               mandarin; neroli; lavender; orange blossom; ja...
concentration                                           eau-de-parfum
intensity                                                         3.0
seasons                                        spring; summer; autumn
times_of_day        

In [34]:
print(df.iloc[752])

id                                            armaf-connoisseur-woman
name                                                Connoisseur Woman
house                                                           Armaf
url                 https://www.scento.com/perfume-database/perfum...
family                                                    White Bloom
secondary_family                                                  NaN
character                               floral; creamy; warm; citrusy
top_notes                                    bergamot; mandarin; pear
heart_notes                    orange blossom; jasmine sambac; neroli
base_notes                  musk; vanilla; benzoin; amber; tonka bean
notes               bergamot; mandarin; pear; orange blossom; jasm...
concentration                                           eau-de-parfum
intensity                                                         3.0
seasons                                        spring; summer; autumn
times_of_day        

In [35]:
print(df.iloc[337])

id                                              guerlain-mon-guerlain
name                                                     Mon Guerlain
house                                                        Guerlain
url                 https://www.scento.com/perfumes/mon-guerlain-g...
family                                                    White Bloom
secondary_family                                        Sunlit Citrus
character                              floral; fresh; citrusy; creamy
top_notes                                    bergamot; mandarin; pear
heart_notes                          neroli; jasmine sambac; lavender
base_notes                                  iris; vanilla; sandalwood
notes               bergamot; mandarin; pear; neroli; jasmine samb...
concentration                                           eau-de-parfum
intensity                                                         3.0
seasons                                        spring; summer; autumn
times_of_day        

Avec ce premier test, on vérifie que sur la base d'un parfum, le modèle est capable de retrouver des parfums similaires. En effet, en regardant les deux premières recommandations de similarité, on constate que le Libre, le Connoisseur Woman et le Mon Guerlain ont des notes similaires : famille White Bloom, les mêmes notes, le même moment de la journée où le porter... 

Le modèle semble bon sur les recommandations.

### Première recommandation sur base d'un profil 

On peut maintenant essayer de faire une recommandation de parfum avec une entrée utilisateur simulée comme elle le serait dans notre future interface.

In [38]:
# Entree utilisateur
notes_aimees = ["vanilla", "jasmine", "musk"]
notes_detestees = ["patchouli"]
moment = "evening"

# Vecteur de preference avec les notes appreciees
vecteurs_notes = [model.wv[note] for note in notes_aimees]
vecteur_utilisateur = np.mean(vecteurs_notes, axis=0)
print("Dimensions du profil utilisateur :", vecteur_utilisateur.shape)

# Comparaison entre le profil et les parfums
scores = cosine_similarity(vecteur_utilisateur.reshape(1, -1),vecteurs_parfums)[0]

# On ajoute les critères du questionnaire
parfum_soiree = df["time_evening"] == 1
parfum_patchouli = df[["top_notes", "heart_notes", "base_notes"]].apply(lambda colonne: colonne.str.contains("patchouli", case=False, na=False)).any(axis=1)
parfum_final = parfum_soiree & ~parfum_patchouli # le ~ permet d'inverser -> on garde les parfums qui ne contiennent pas de patchoulis

# Recommandations finales
resultats = df.loc[parfum_final, ["name", "house"]].copy()
resultats["similarite"] = scores[parfum_final]
resultats = resultats.sort_values(by="similarite", ascending=False)
resultats.head(10)

Dimensions du profil utilisateur : (100,)


/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,name,house,similarite
679,Supremacy Gold,Afnan,0.834094
1036,Wind Flowers,Creed,0.817240
671,Rare Passion,Afnan,0.816772
1468,Libre Intense,Yves Saint Laurent,0.813290
1135,Cruel Gardénia,Guerlain,0.812628
913,My Way Intense,Giorgio Armani,0.810038
1302,all of me intense,Narciso Rodriguez,0.809459
1139,Samsara,Guerlain,0.807445
888,Vanity Femme Essence Woman,Armaf,0.805160
1281,Vanilla Extasy :,Montale,0.802268


In [39]:
# Verifications
verification = df.loc[resultats.head(10).index, ["name", "top_notes", "heart_notes", "base_notes", "time_evening"]].copy()
verification["similarite"] = resultats.head(10)["similarite"]
verification

,name,top_notes,heart_notes,base_notes,time_evening,similarite
679,Supremacy Gold,ylang-ylang; neroli,vetiver; musk; vanilla,jasmine; musk; sandalwood,1,0.834094
1036,Wind Flowers,jasmine; orange blossom; peach,jasmine; tuberose; rose,musk; praline; sandalwood; iris,1,0.817240
671,Rare Passion,orange; musk; rose,amber; wood; vanilla,rose; musk; orange blossom,1,0.816772
1468,Libre Intense,lavender; orange blossom; orchid,jasmine sambac; orange blossom; orchid,tonka bean; vanilla; ambergris; vetiver,1,0.813290
1135,Cruel Gardénia,rose; peach; neroli,gardenia; ylang-ylang; musk,tonka bean; vanilla; sandalwood,1,0.812628
913,My Way Intense,bergamot; orange blossom,tuberose; jasmine,cedarwood; vanilla; musk,1,0.810038
1302,all of me intense,iris; jasmine sambac,rose; geranium; tuberose,musk; vanilla; sandalwood,1,0.809459
1139,Samsara,honeysuckle,jasmine; ylang-ylang,sandalwood; iris; tonka bean; vanilla,1,0.807445
888,Vanity Femme Essence Woman,freesia; orange blossom; peach,jasmine; magnolia; rose,amber; musk; vanilla,1,0.805160
1281,Vanilla Extasy :,bergamot; peach; apricot,lavender; jasmine; ylang-ylang,sandalwood; vanilla; benzoin; amber; musk,1,0.802268


Le premier test est concluant : les parfums choisis par le modèle sont bien des parfums de soir, contenant de la vanille et/ou du musc et/ou du jasmin, et ne contiennent pas de patchouli.

### Conclusion

Pour conclure sur ce modèle, on peut dire que le modèle Word2Vec semble capable de sortir des recommandations adaptées à une demande utilisateur. Il faudra maintenant comparer les modèles pour garder celui qui fait de meilleure recommandations.